# 05 分層分析與干擾因子 — 參考解答

松柏護理之家退伍軍人症群聚事件分層分析練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import chi2_contingency
from epi_learning.metrics import risk_ratio

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

## 題目 1：水療使用的干擾分析

In [ ]:
# --- 粗 RR ---
ct_hydro = pd.crosstab(df["hydrotherapy_use"], df["infected"])
a0 = int(ct_hydro.loc[1, 1])
b0 = int(ct_hydro.loc[1, 0])
c0 = int(ct_hydro.loc[0, 1])
d0 = int(ct_hydro.loc[0, 0])
crude_rr_hydro = risk_ratio(a0, a0 + b0, c0, c0 + d0)
print(f"粗 RR (hydrotherapy → infected) = {crude_rr_hydro:.3f}")

# --- 驗證干擾條件 ---
print("\n=== 功能狀態 × 水療使用率 ===")
print(pd.crosstab(df["functional_status"], df["hydrotherapy_use"],
                  normalize="index").round(3))

# --- 分層 RR ---
strata = sorted(df["functional_status"].unique())
hydro_results = []

for s in strata:
    sub = df[df["functional_status"] == s]
    ct_s = pd.crosstab(sub["hydrotherapy_use"], sub["infected"])
    if ct_s.shape != (2, 2):
        print(f"  {s}: 跳過")
        continue
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    hydro_results.append({
        "stratum": s, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

hydro_df = pd.DataFrame(hydro_results)
print("\n=== 分層 RR ===")
for _, row in hydro_df.iterrows():
    print(f"  {row['stratum']:20s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})")
print(f"\n  粗 RR = {crude_rr_hydro:.3f}")

## 題目 2：Mantel-Haenszel 調整

In [ ]:
numerator = 0
denominator = 0

for _, row in hydro_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh_hydro = numerator / denominator

print(f"Mantel-Haenszel 調整後 RR = {rr_mh_hydro:.3f}")
print(f"粗 RR                     = {crude_rr_hydro:.3f}")
print(f"差異                      = {crude_rr_hydro - rr_mh_hydro:.3f}")

if abs(crude_rr_hydro - rr_mh_hydro) > 0.1:
    print("\n→ 功能狀態確實是水療使用的干擾因子（粗 RR 被膨脹）")
else:
    print("\n→ 控制功能狀態後 RR 變化不大，干擾效應有限")

## 題目 3（挑戰題）：按年齡組分層 + 森林圖

In [ ]:
# 粗 RR
ct_shower = pd.crosstab(df["shower_use"], df["infected"])
a_crude = int(ct_shower.loc[1, 1])
b_crude = int(ct_shower.loc[1, 0])
c_crude = int(ct_shower.loc[0, 1])
d_crude = int(ct_shower.loc[0, 0])
crude_rr = risk_ratio(a_crude, a_crude + b_crude, c_crude, c_crude + d_crude)

# 建立年齡組
df["age_group"] = pd.cut(
    df["age"], bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

# 分層 RR
age_results = []
for grp in ["60-69", "70-79", "80-89", "90+"]:
    sub = df[df["age_group"] == grp]
    ct_s = pd.crosstab(sub["shower_use"], sub["infected"])
    if ct_s.shape != (2, 2):
        print(f"  {grp}: 跳過")
        continue
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    age_results.append({
        "stratum": grp, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

age_df = pd.DataFrame(age_results)
print("=== 按年齡組分層 RR ===")
for _, row in age_df.iterrows():
    print(f"  {row['stratum']:10s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})  n={row['n']}")

In [ ]:
# 森林圖
fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(age_df))

ax.errorbar(
    age_df["RR"], y_pos,
    xerr=[age_df["RR"] - age_df["CI_lower"],
          age_df["CI_upper"] - age_df["RR"]],
    fmt="o", color="#2c7fb8", capsize=4, markersize=8,
)
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=crude_rr, color="red", linestyle=":", alpha=0.7,
           label=f"粗 RR={crude_rr:.2f}")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(age_df["stratum"])
ax.set_xlabel("Risk Ratio (RR)")
ax.set_title("森林圖：淋浴使用 → 感染（按年齡組分層）")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# MH 調整後 RR
num = 0
den = 0
for _, row in age_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    num += a_i * (c_i + d_i) / n_i
    den += c_i * (a_i + b_i) / n_i

rr_mh_age = num / den

print(f"MH 調整後 RR（控制年齡） = {rr_mh_age:.3f}")
print(f"粗 RR                    = {crude_rr:.3f}")
print(f"差異                     = {crude_rr - rr_mh_age:.3f}")

# 同質性
rr_vals = age_df["RR"].values
print(f"\n各層 RR 範圍：{rr_vals.min():.3f} – {rr_vals.max():.3f}")
if rr_vals.max() - rr_vals.min() > 0.5:
    print("→ 各年齡組 RR 差異較大，可能存在年齡的效果修飾")
else:
    print("→ 各年齡組 RR 相近，年齡的交互作用不明顯")

### 解讀

- **功能狀態**：臥床住民不淋浴也較少感染，能行走的住民淋浴率高也較多感染 → 經典干擾
- **MH 調整後**：如果 RR_MH 明顯小於粗 RR，確認功能狀態是干擾因子
- **年齡分層**：如果各年齡組的 RR 接近，年齡交互作用不大
- **限制**：分層分析一次只能控制一個變項 → 需要 Ch06 邏輯斯迴歸同時調整多個因子

## 題目 4 解答

In [ ]:
# --- 資料：COVID-19 病例的年齡與共病 ---
rng = np.random.default_rng(20)
n = 800

age_group = rng.choice(["under60", "60plus"], size=n, p=[0.6, 0.4])
comorbidity = np.array([
    rng.binomial(1, 0.5 if ag == "60plus" else 0.15) for ag in age_group
])
death_prob = np.select(
    [
        (age_group == "under60") & (comorbidity == 0),
        (age_group == "under60") & (comorbidity == 1),
        (age_group == "60plus") & (comorbidity == 0),
        (age_group == "60plus") & (comorbidity == 1),
    ],
    [0.02, 0.05, 0.12, 0.30],
)
death = rng.binomial(1, death_prob)

covid_df = pd.DataFrame({
    "case_id": [f"C{i:04d}" for i in range(n)],
    "age_group": age_group,
    "comorbidity": comorbidity,
    "death": death,
})

# --- 粗 RR ---
ct_covid = pd.crosstab(covid_df["comorbidity"], covid_df["death"])
a0 = int(ct_covid.loc[1, 1])
b0 = int(ct_covid.loc[1, 0])
c0 = int(ct_covid.loc[0, 1])
d0 = int(ct_covid.loc[0, 0])
crude_rr_covid = risk_ratio(a0, a0 + b0, c0, c0 + d0)
print(f"粗 RR (comorbidity → death) = {crude_rr_covid:.3f}")

# --- 驗證干擾條件 ---
print("\n=== 年齡層 × 共病比例 ===")
print(pd.crosstab(covid_df["age_group"], covid_df["comorbidity"], normalize="index").round(3))
print("\n=== 年齡層 × 死亡比例 ===")
print(pd.crosstab(covid_df["age_group"], covid_df["death"], normalize="index").round(3))

# --- 分層 RR ---
covid_results = []
for ag in ["under60", "60plus"]:
    sub = covid_df[covid_df["age_group"] == ag]
    ct_s = pd.crosstab(sub["comorbidity"], sub["death"])
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    covid_results.append({
        "stratum": ag, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

covid_strat_df = pd.DataFrame(covid_results)
print("\n=== 按年齡層分層 RR ===")
for _, row in covid_strat_df.iterrows():
    print(f"  {row['stratum']:10s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})")
print(f"\n  粗 RR = {crude_rr_covid:.3f}")

# --- Mantel-Haenszel 調整後 RR ---
numerator = 0
denominator = 0
for _, row in covid_strat_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh_covid = numerator / denominator
print(f"\nMantel-Haenszel 調整後 RR = {rr_mh_covid:.3f}")
print(f"粗 RR                     = {crude_rr_covid:.3f}")
print(f"差異                      = {crude_rr_covid - rr_mh_covid:.3f}")

if crude_rr_covid - rr_mh_covid > 0.5:
    print("\n→ 年齡是共病影響死亡率的干擾因子：高齡族群共病比例高、死亡風險也高，膨脹了粗 RR")
else:
    print("\n→ 控制年齡後 RR 變化不大，干擾效應有限")

## 題目 5 解答

In [ ]:
# --- 資料：流感疫苗接種與年齡 ---
rng = np.random.default_rng(11)
n = 800

age_group = rng.choice(["under65", "65plus"], size=n, p=[0.7, 0.3])
vaccinated = np.array([
    rng.binomial(1, 0.7 if ag == "65plus" else 0.3) for ag in age_group
])
infect_prob = np.select(
    [
        (age_group == "under65") & (vaccinated == 0),
        (age_group == "under65") & (vaccinated == 1),
        (age_group == "65plus") & (vaccinated == 0),
        (age_group == "65plus") & (vaccinated == 1),
    ],
    [0.20, 0.10, 0.40, 0.20],
)
infected = rng.binomial(1, infect_prob)

flu_df = pd.DataFrame({
    "case_id": [f"F{i:04d}" for i in range(n)],
    "age_group": age_group,
    "vaccinated": vaccinated,
    "infected": infected,
})

# --- 粗 RR ---
ct_flu = pd.crosstab(flu_df["vaccinated"], flu_df["infected"])
a0 = int(ct_flu.loc[1, 1])
b0 = int(ct_flu.loc[1, 0])
c0 = int(ct_flu.loc[0, 1])
d0 = int(ct_flu.loc[0, 0])
crude_rr_flu = risk_ratio(a0, a0 + b0, c0, c0 + d0)
print(f"粗 RR (vaccinated → infected) = {crude_rr_flu:.3f}")

# --- 驗證干擾條件 ---
print("\n=== 年齡層 × 疫苗接種率 ===")
print(pd.crosstab(flu_df["age_group"], flu_df["vaccinated"], normalize="index").round(3))
print("\n=== 年齡層 × 感染率 ===")
print(pd.crosstab(flu_df["age_group"], flu_df["infected"], normalize="index").round(3))

# --- 分層 RR ---
flu_results = []
for ag in ["under65", "65plus"]:
    sub = flu_df[flu_df["age_group"] == ag]
    ct_s = pd.crosstab(sub["vaccinated"], sub["infected"])
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    flu_results.append({
        "stratum": ag, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

flu_strat_df = pd.DataFrame(flu_results)
print("\n=== 按年齡層分層 RR ===")
for _, row in flu_strat_df.iterrows():
    print(f"  {row['stratum']:10s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})")
print(f"\n  粗 RR = {crude_rr_flu:.3f}")

# --- Mantel-Haenszel 調整後 RR ---
numerator = 0
denominator = 0
for _, row in flu_strat_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh_flu = numerator / denominator
print(f"\nMantel-Haenszel 調整後 RR = {rr_mh_flu:.3f}")
print(f"粗 RR                     = {crude_rr_flu:.3f}")

print("\n→ 粗 RR 比 MH RR 更接近 1，是因為年齡同時提高接種率與感染風險（適應症干擾），")
print("  掩蓋了疫苗的保護效果；MH 調整後才更準確反映疫苗的保護效果（RR 更小、更保護）。")

## 題目 6 解答

In [ ]:
# --- 資料：甲型肝炎聚餐群聚事件 ---
rng = np.random.default_rng(5)
n = 900

vaccinated = rng.binomial(1, 0.35, size=n)
ate_shellfish = np.array([
    rng.binomial(1, 0.3 if v == 1 else 0.6) for v in vaccinated
])
infect_prob = np.select(
    [
        (vaccinated == 0) & (ate_shellfish == 0),
        (vaccinated == 0) & (ate_shellfish == 1),
        (vaccinated == 1) & (ate_shellfish == 0),
        (vaccinated == 1) & (ate_shellfish == 1),
    ],
    [0.05, 0.35, 0.01, 0.07],
)
infected = rng.binomial(1, infect_prob)

hav_df = pd.DataFrame({
    "case_id": [f"H{i:04d}" for i in range(n)],
    "vaccinated": vaccinated,
    "ate_shellfish": ate_shellfish,
    "infected": infected,
})

# --- 粗 RR ---
ct_hav = pd.crosstab(hav_df["ate_shellfish"], hav_df["infected"])
a0 = int(ct_hav.loc[1, 1])
b0 = int(ct_hav.loc[1, 0])
c0 = int(ct_hav.loc[0, 1])
d0 = int(ct_hav.loc[0, 0])
crude_rr_hav = risk_ratio(a0, a0 + b0, c0, c0 + d0)
print(f"粗 RR (ate_shellfish → infected) = {crude_rr_hav:.3f}")

# --- 驗證干擾條件 ---
print("\n=== 疫苗接種史 × 生蠔食用率 ===")
print(pd.crosstab(hav_df["vaccinated"], hav_df["ate_shellfish"], normalize="index").round(3))
print("\n=== 疫苗接種史 × 感染率 ===")
print(pd.crosstab(hav_df["vaccinated"], hav_df["infected"], normalize="index").round(3))

# --- 分層 RR ---
hav_results = []
for v in [0, 1]:
    sub = hav_df[hav_df["vaccinated"] == v]
    ct_s = pd.crosstab(sub["ate_shellfish"], sub["infected"])
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    hav_results.append({
        "stratum": "vaccinated" if v == 1 else "unvaccinated", "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

hav_strat_df = pd.DataFrame(hav_results)
print("\n=== 按疫苗接種史分層 RR ===")
for _, row in hav_strat_df.iterrows():
    print(f"  {row['stratum']:12s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})")
print(f"\n  粗 RR = {crude_rr_hav:.3f}")

# --- Mantel-Haenszel 調整後 RR ---
numerator = 0
denominator = 0
for _, row in hav_strat_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh_hav = numerator / denominator
print(f"\nMantel-Haenszel 調整後 RR = {rr_mh_hav:.3f}")
print(f"粗 RR                     = {crude_rr_hav:.3f}")
print(f"差異                      = {crude_rr_hav - rr_mh_hav:.3f}")

if crude_rr_hav - rr_mh_hav > 0.5:
    print("\n→ 疫苗接種史是干擾因子：未接種者較常食用生蠔、感染風險也較高，膨脹了粗 RR")
else:
    print("\n→ 控制疫苗接種史後 RR 變化不大，干擾效應有限")

## 題目 7 解答

In [ ]:
# --- 資料：登革熱跨區域調查 ---
rng = np.random.default_rng(42)
n = 900

region = rng.choice(["urban", "suburban", "rural"], size=n, p=[0.4, 0.35, 0.25])
water_p = {"urban": 0.2, "suburban": 0.4, "rural": 0.6}
standing_water = np.array([rng.binomial(1, water_p[r]) for r in region])
infect_p = {
    ("urban", 0): 0.03, ("urban", 1): 0.09,
    ("suburban", 0): 0.08, ("suburban", 1): 0.24,
    ("rural", 0): 0.15, ("rural", 1): 0.45,
}
infect_prob = np.array([infect_p[(r, w)] for r, w in zip(region, standing_water)])
infected = rng.binomial(1, infect_prob)

dengue_df = pd.DataFrame({
    "case_id": [f"D{i:04d}" for i in range(n)],
    "region": region,
    "standing_water": standing_water,
    "infected": infected,
})

# --- 粗 RR ---
ct_dengue = pd.crosstab(dengue_df["standing_water"], dengue_df["infected"])
a0 = int(ct_dengue.loc[1, 1])
b0 = int(ct_dengue.loc[1, 0])
c0 = int(ct_dengue.loc[0, 1])
d0 = int(ct_dengue.loc[0, 0])
crude_rr_dengue = risk_ratio(a0, a0 + b0, c0, c0 + d0)
print(f"粗 RR (standing_water → infected) = {crude_rr_dengue:.3f}")

# --- 驗證干擾條件 ---
print("\n=== 區域 × 積水比例 ===")
print(pd.crosstab(dengue_df["region"], dengue_df["standing_water"], normalize="index").round(3))
print("\n=== 區域 × 感染比例 ===")
print(pd.crosstab(dengue_df["region"], dengue_df["infected"], normalize="index").round(3))

# --- 分層 RR ---
dengue_results = []
for r_ in ["urban", "suburban", "rural"]:
    sub = dengue_df[dengue_df["region"] == r_]
    ct_s = pd.crosstab(sub["standing_water"], sub["infected"])
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    dengue_results.append({
        "stratum": r_, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

dengue_strat_df = pd.DataFrame(dengue_results)
print("\n=== 按區域分層 RR ===")
for _, row in dengue_strat_df.iterrows():
    print(f"  {row['stratum']:10s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})  n={row['n']}")
print(f"\n  粗 RR = {crude_rr_dengue:.3f}")

# --- Mantel-Haenszel 調整後 RR ---
numerator = 0
denominator = 0
for _, row in dengue_strat_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh_dengue = numerator / denominator
print(f"\nMantel-Haenszel 調整後 RR = {rr_mh_dengue:.3f}")
print(f"粗 RR                     = {crude_rr_dengue:.3f}")
print(f"差異                      = {crude_rr_dengue - rr_mh_dengue:.3f}")

rr_vals = dengue_strat_df["RR"].values
print(f"\n各區域 RR 範圍：{rr_vals.min():.3f} – {rr_vals.max():.3f}")
if crude_rr_dengue - rr_mh_dengue > 0.5:
    print("→ 區域是積水暴露與感染登革熱關聯的干擾因子：鄉村地區積水比例高、感染風險也高，膨脹了粗 RR")
else:
    print("→ 控制區域後 RR 變化不大，干擾效應有限")

## 題目 8 解答

In [ ]:
# --- 資料：結核病接觸者篩檢 ---
from epi_learning.metrics import odds_ratio

rng = np.random.default_rng(291)
n = 900

diabetes = rng.binomial(1, 0.25, size=n)
close_contact = np.array([
    rng.binomial(1, 0.5 if d == 1 else 0.25) for d in diabetes
])
tb_prob = np.select(
    [
        (diabetes == 0) & (close_contact == 0),
        (diabetes == 0) & (close_contact == 1),
        (diabetes == 1) & (close_contact == 0),
        (diabetes == 1) & (close_contact == 1),
    ],
    [0.02, 0.10, 0.08, 0.32],
)
active_tb = rng.binomial(1, tb_prob)

tb_df = pd.DataFrame({
    "case_id": [f"T{i:04d}" for i in range(n)],
    "diabetes": diabetes,
    "close_contact": close_contact,
    "active_tb": active_tb,
})

# --- 粗 OR ---
ct_tb = pd.crosstab(tb_df["close_contact"], tb_df["active_tb"])
a0 = int(ct_tb.loc[1, 1])
b0 = int(ct_tb.loc[1, 0])
c0 = int(ct_tb.loc[0, 1])
d0 = int(ct_tb.loc[0, 0])
crude_or_tb = odds_ratio(a0, b0, c0, d0)
print(f"粗 OR (close_contact → active_tb) = {crude_or_tb:.3f}")

# --- 驗證干擾條件 ---
print("\n=== 糖尿病 × 密切接觸比例 ===")
print(pd.crosstab(tb_df["diabetes"], tb_df["close_contact"], normalize="index").round(3))
print("\n=== 糖尿病 × 活動性結核比例 ===")
print(pd.crosstab(tb_df["diabetes"], tb_df["active_tb"], normalize="index").round(3))

# --- 分層 OR ---
tb_results = []
for d_ in [0, 1]:
    sub = tb_df[tb_df["diabetes"] == d_]
    ct_s = pd.crosstab(sub["close_contact"], sub["active_tb"])
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    or_s = odds_ratio(a_s, b_s, c_s, d_s)
    ln_or = np.log(or_s)
    se = np.sqrt(1/a_s + 1/b_s + 1/c_s + 1/d_s)
    ci_lo = np.exp(ln_or - 1.96 * se)
    ci_hi = np.exp(ln_or + 1.96 * se)
    tb_results.append({
        "stratum": "diabetic" if d_ == 1 else "non_diabetic", "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "OR": or_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

tb_strat_df = pd.DataFrame(tb_results)
print("\n=== 按糖尿病狀態分層 OR ===")
for _, row in tb_strat_df.iterrows():
    print(f"  {row['stratum']:14s}  OR={row['OR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})  n={row['n']}")
print(f"\n  粗 OR = {crude_or_tb:.3f}")

# --- 森林圖 ---
fig, ax = plt.subplots(figsize=(8, 3.5))
y_pos = range(len(tb_strat_df))

ax.errorbar(
    tb_strat_df["OR"], y_pos,
    xerr=[tb_strat_df["OR"] - tb_strat_df["CI_lower"],
          tb_strat_df["CI_upper"] - tb_strat_df["OR"]],
    fmt="o", color="#2c7fb8", capsize=4, markersize=8,
)
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=crude_or_tb, color="red", linestyle=":", alpha=0.7,
           label=f"粗 OR={crude_or_tb:.2f}")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(tb_strat_df["stratum"])
ax.set_xlabel("Odds Ratio (OR)")
ax.set_title("森林圖：密切接觸 → 活動性結核（按糖尿病狀態分層）")
ax.legend()
plt.tight_layout()
plt.show()

# --- Mantel-Haenszel 調整後 OR ---
numerator = 0
denominator = 0
for _, row in tb_strat_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * d_i / n_i
    denominator += b_i * c_i / n_i

or_mh_tb = numerator / denominator
print(f"\nMantel-Haenszel 調整後 OR = {or_mh_tb:.3f}")
print(f"粗 OR                     = {crude_or_tb:.3f}")
print(f"差異                      = {crude_or_tb - or_mh_tb:.3f}")

or_vals = tb_strat_df["OR"].values
print(f"\n各層 OR 範圍：{or_vals.min():.3f} – {or_vals.max():.3f}")
if or_vals.max() - or_vals.min() > 1.5:
    print("→ 各層 OR 差異較大，糖尿病可能是效果修飾因子（effect modifier）")
else:
    print("→ 各層 OR 相近，糖尿病較可能是單純的干擾因子，而非效果修飾因子")

if crude_or_tb - or_mh_tb > 1.0:
    print("→ 糖尿病同時符合干擾條件：糖尿病患者密切接觸比例較高、結核風險也較高，膨脹了粗 OR")
else:
    print("→ 控制糖尿病後 OR 變化不大，干擾效應有限")